# RL Racing — reproducible Kaggle training

Private, headless PPO training on the exact packaged source. Run acceptance before full training. Saved notebook outputs contain verified exports; live files alone are not persistent. CPU is intentional. Review the accountâ€™s current session limit and reserve setup time before choosing MAX_SESSION_SECONDS.


In [ ]:
MODE = 'acceptance'
MAX_SESSION_SECONDS = 39600
RESUME_ARCHIVE = None
CONDITION = None
SPLIT_MANIFEST_SHA256 = None


## 1. Verify attached source and copy into writable working storage


In [ ]:
import hashlib
import json
import pathlib
import re
import subprocess
import sys
import zipfile

INPUT = pathlib.Path('/kaggle/input')
ROOT = pathlib.Path('/kaggle/working/ai-racing')
manifests = [p for p in INPUT.rglob('source-manifest.json') if p.with_name('source.zip').is_file() or p.with_name('source').is_dir()]
assert len(manifests) == 1, f'Attach exactly one versioned source dataset: {manifests}'
source_manifest = json.loads(manifests[0].read_text())
archive = manifests[0].with_name('source.zip')
assert not ROOT.exists(), 'Use a fresh session/working directory'
if archive.is_file():
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == source_manifest['archive_sha256']
    with zipfile.ZipFile(archive) as package:
        for entry in package.infolist():
            target = (ROOT / entry.filename).resolve()
            assert target.is_relative_to(ROOT.resolve()) and ':' not in entry.filename
        package.extractall(ROOT)
else:
    # Kaggle can expand uploaded .zip files into a source/ input directory.
    import shutil

    expanded = manifests[0].parent/'source'
    assert expanded.is_dir(), 'Missing archived or expanded source input'
    actual = {p.relative_to(expanded).as_posix() for p in expanded.rglob('*') if p.is_file()}
    assert actual == set(source_manifest['files']), 'Expanded source file coverage mismatch'
    for name, digest in source_manifest['files'].items():
        target = (ROOT/name).resolve()
        assert target.is_relative_to(ROOT.resolve()) and ':' not in name
        assert hashlib.sha256((expanded/name).read_bytes()).hexdigest() == digest, name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(expanded/name, target)
for name, digest in source_manifest['files'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == digest, name
print('Verified source:', source_manifest['source_sha256'])
print('Input version:', str(manifests[0].parent))


## 2. Provision isolated Python 3.11.16 and install the hash-locked CPU runtime

Internet must be enabled. Every project command uses the explicit isolated interpreter; the notebook kernelâ€™s packages are not used for training.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'uv==0.13.1'], check=True)
UV = [sys.executable, '-m', 'uv']
subprocess.run(UV + ['python', 'install', '3.11.16'], check=True)
subprocess.run(UV + ['venv', str(ROOT/'.runtime'), '--python', '3.11.16', '--seed'], check=True)
PYTHON = str(ROOT/'.runtime/bin/python')
subprocess.run(UV + ['pip', 'sync', str(ROOT/'requirements-kaggle.lock.txt'), '--python', PYTHON,
                     '--require-hashes', '--torch-backend', 'cpu'], check=True)
subprocess.run(UV + ['pip', 'install', '--python', PYTHON, '--no-deps', '--no-build-isolation', '-e', str(ROOT)], check=True)
def run(*args, allowed=(0,)):
    result = subprocess.run([PYTHON, *map(str,args)], cwd=ROOT)
    if result.returncode not in allowed:
        raise subprocess.CalledProcessError(result.returncode, result.args)
    return result.returncode
run('-c', "import sys,platform,importlib.metadata as m; assert platform.python_version()=='3.11.16'; import racing.training.train; assert 'pygame' not in sys.modules; print(sys.executable); print({p:m.version(p) for p in ['torch','stable_baselines3','gymnasium','tensorboard']})")
locked = dict(re.findall(r'^([A-Za-z0-9_-]+)==(\S+)', (ROOT/'requirements-kaggle.lock.txt').read_text(), re.M))
run('-c', 'import importlib.metadata as m; pins='+repr(locked)+'; mismatches=[(p,v,m.version(p)) for p,v in pins.items() if m.version(p)!=v]; assert not mismatches,mismatches; print("All lock versions match")')
assert hashlib.sha256((ROOT/'requirements-kaggle.lock.txt').read_bytes()).hexdigest() == source_manifest['lockfile_sha256']


## 3. Verify contracts and resolve execution settings


In [ ]:
def read_yaml(path):
    return json.loads(subprocess.check_output([PYTHON, "-c", "import json,sys,yaml; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))", str(path)], cwd=ROOT, text=True))

run('-m', 'pytest', 'tests/unit/test_training_contracts.py', 'tests/unit/test_training_export.py', 'tests/integration/test_policy.py', '-q')
execution = read_yaml(ROOT/'configs/kaggle.yaml')
assert MAX_SESSION_SECONDS > execution['shutdown_margin_seconds']
execution['max_session_seconds'] = MAX_SESSION_SECONDS
EXECUTION = ROOT/'execution.session.yaml'
EXECUTION.write_text(json.dumps(execution))
run('-c', "from racing.training.manifest import source_identity; assert source_identity()=="+repr(source_manifest['source_sha256']))
(Root_provenance := ROOT/'kaggle_input_provenance.json').write_text(json.dumps({
    'source': source_manifest, 'input_path': str(manifests[0].parent),
    'mode': MODE, 'session_seconds': MAX_SESSION_SECONDS}, indent=2))
SMOKE = ROOT/'configs/train_ppo.yaml'
smoke = read_yaml(SMOKE)
smoke.update(total_timesteps=2048, n_steps=1024, eval_every_steps=1024)
SMOKE_CONFIG = ROOT/'smoke.session.yaml'
SMOKE_CONFIG.write_text(json.dumps(smoke))

if MODE.startswith('phase04'):
    run('-m', 'pytest', 'tests/unit/test_phase04_learning.py', 'tests/unit/test_validation_integrity.py', '-q')
    assert CONDITION in ('single', 'multi') and SPLIT_MANIFEST_SHA256
    assert source_manifest.get('split_manifest_sha256') == SPLIT_MANIFEST_SHA256, 'Packaged split identity mismatch'
    # Validate the frozen dataset; never generate tracks in a training session.
    run('-c', "from racing.tracks.splits import load_manifest; m=load_manifest('experiments/splits/v1.json'); assert m['manifest_sha256']=="+repr(SPLIT_MANIFEST_SHA256))
    suffix = '_smoke' if MODE == 'phase04-smoke' else ''
    condition_path = ROOT/f'experiments/phase04_{CONDITION}{suffix}.yaml'
    condition_config = read_yaml(condition_path)
    assert condition_config['condition'] == CONDITION
    PROFILE = ROOT/'execution-profile.phase04.json'
    run('-c', "from racing.training.manifest import dependencies,source_identity,file_hash,ROOT,write_json; d=dependencies(); write_json('execution-profile.phase04.json',dict(platform='kaggle',device='cpu',torch_threads=1,python_version=d['python'],packages=d['packages'],source_sha256=source_identity(),lockfile_sha256=file_hash(ROOT/'requirements-kaggle.lock.txt')))")
    condition_config['execution_profile_registration'] = str(PROFILE)
    SMOKE = ROOT/'phase04.session.yaml'
    SMOKE.write_text(json.dumps(condition_config))
    shutil_provenance = ROOT/'kaggle_input_provenance.json'
    provenance = json.loads(shutil_provenance.read_text())
    provenance.update(condition=CONDITION, split_manifest_sha256=SPLIT_MANIFEST_SHA256,
                      condition_config=str(condition_path.relative_to(ROOT)), execution_profile=json.loads(PROFILE.read_text()))
    shutil_provenance.write_text(json.dumps(provenance, indent=2))


## 4. Train or resume through the shared CLI

Exit 75 is a safe session stop. All other nonzero exits are errors. A stop preserves the original training target and does not create an extra selectable validation checkpoint.


In [ ]:
TRAIN = ['scripts/train.py', '--execution-config', str(EXECUTION)]
active_run = None
if MODE == 'acceptance':
    active_run = ROOT/'runs/kaggle-smoke'
    run(*TRAIN, '--config', SMOKE_CONFIG, '--run-dir', active_run)
    controlled = dict(smoke, total_timesteps=4096, n_steps=512, eval_every_steps=2048)
    CONTROL = ROOT/'control.session.yaml'
    CONTROL.write_text(json.dumps(controlled))
    stopped = ROOT/'runs/session-stop'
    status = run(*TRAIN, '--config', CONTROL, '--run-dir', stopped, '--stop-after-updates', '1', allowed=(75,))
    assert status == 75
    run('scripts/export_training.py', 'export', stopped, '--destination', '/kaggle/working/session-stop.zip')
    (pathlib.Path('/kaggle/working')/'resume-config.yaml').write_text(json.dumps(controlled))
elif MODE == 'resume-acceptance':
    matches = list(INPUT.rglob(RESUME_ARCHIVE or 'session-stop.zip'))
    assert len(matches)==1, matches
    imported = ROOT/'imported-parent'
    run('scripts/export_training.py', 'import', matches[0], '--destination', imported)
    inventory = json.loads((imported/'checkpoints/committed.json').read_text())
    checkpoint = imported/inventory['latest']
    parent_manifest = json.loads((imported/'manifest.json').read_text())
    resumed_config = ROOT/'resume.session.yaml'
    resumed_config.write_text(json.dumps(parent_manifest['training_config']))
    active_run = ROOT/'runs/kaggle-resumed'
    run(*TRAIN, '--config', resumed_config, '--run-dir', active_run, '--resume', checkpoint)
    result = json.loads((active_run/'manifest.json').read_text())
    assert result['status']=='completed' and result['requested_transitions']==4096 and result['actual_transitions']==4096
    ledger = json.loads((active_run/'validation_ledger.json').read_text())
    assert [entry['checkpoint_step'] for entry in ledger]==[2048,4096], ledger
else:
    assert MODE in ('full', 'phase04-smoke', 'phase04-full')
    active_run = ROOT/(f'runs/phase04-{CONDITION}' if MODE.startswith('phase04') else 'runs/ppo-demo-seed42')
    extra = []
    if RESUME_ARCHIVE:
        matches = list(INPUT.rglob(RESUME_ARCHIVE))
        assert len(matches)==1, matches
        imported = ROOT/'imported-parent'
        run('scripts/export_training.py', 'import', matches[0], '--destination', imported)
        checkpoint = subprocess.check_output([PYTHON, 'scripts/export_training.py', 'checkpoint', str(imported)], cwd=ROOT, text=True).strip()
        extra = ['--resume', checkpoint]
    status = run(*TRAIN, '--config', SMOKE, '--run-dir', active_run, *extra, allowed=(0,75))
    print('Training completed' if status==0 else 'Safely interrupted; resume the exported bundle')


## 5. Headless evaluation and identical-protocol baseline comparison


In [ ]:
manifest = json.loads((active_run/'manifest.json').read_text())
if manifest['status']=='completed' and MODE.startswith('phase04'):
    run('scripts/evaluate.py', '--agent', 'ppo', '--model', active_run/'best_model.zip',
        '--split', 'validation', '--split-manifest', 'experiments/splits/v1.json',
        '--episodes-per-track', '3', '--seed', '9000', '--output', ROOT/f'runs/phase04-{CONDITION}-validation')
    print('Validation-only results; one training seed is preliminary. Test inference remains locked.')
elif manifest['status']=='completed' and MODE != 'resume-acceptance':
    for agent in ('random','scripted','ppo'):
        extra = ['--model', active_run/'best_model.zip'] if agent=='ppo' else []
        run('scripts/evaluate.py','--agent',agent,'--track','demo','--episodes','5','--seed','9000',
            '--output', ROOT/f'runs/comparison-{agent}', *extra)
    comparison = {agent:json.loads((ROOT/f'runs/comparison-{agent}/summary.json').read_text()) for agent in ('random','scripted','ppo')}
    (ROOT/'runs/baseline-comparison.json').write_text(json.dumps(comparison,indent=2))
    print({agent:{k:s[k] for k in ('completion_rate','mean_progress_fraction','lap_time_median_s')} for agent,s in comparison.items()})
elif manifest['status']=='completed':
    print('Fresh-session resume verified; baseline comparison is retained in the smoke notebook outputs')
else:
    print('Partial training: no completed-run baseline claim')


## 6. Export and verify all committed state and ancestor artifacts

Save this notebook version with outputs, then download the export and its inventory. A live session alone does not guarantee persistence. Deployment requires model.zip plus its sidecar; resume requires the full verified export. Run the desktop watch command locally after downloading.


In [ ]:
import shutil

for name in ('source-manifest.json',):
    shutil.copyfile(manifests[0].parent/name, ROOT/'runs'/name)
shutil.copyfile(ROOT/'requirements-kaggle.lock.txt', ROOT/'runs/requirements-kaggle.lock.txt')
shutil.copyfile(ROOT/'kaggle_input_provenance.json', ROOT/'runs/kaggle_input_provenance.json')
if MODE.startswith('phase04'):
    shutil.copyfile(ROOT/'experiments/splits/v1.json', ROOT/'runs/split-manifest.json')
    shutil.copyfile(ROOT/'execution-profile.phase04.json', ROOT/'runs/execution-profile.phase04.json')
    shutil.copyfile(SMOKE, ROOT/'runs/condition-config.resolved.yaml')
(ROOT/'runs/active-run.json').write_text(json.dumps({'run_directory': str(active_run.relative_to(ROOT/'runs'))}))
run('scripts/export_training.py', 'export', ROOT/'runs', '--destination', '/kaggle/working/training-export.zip')
print('Export SHA-256:', hashlib.sha256(pathlib.Path('/kaggle/working/training-export.zip').read_bytes()).hexdigest())
print('Run status:', json.loads((active_run/'manifest.json').read_text())['status'])
print('Local watch: python scripts/play.py --mode watch --model PATH/best_model.zip --track demo')

# The notebook kernel is separate; retain source/runs but avoid saving the disposable environment.
runtime = (ROOT/".runtime").resolve()
assert runtime.parent == ROOT.resolve()
shutil.rmtree(runtime)
